# 03-2 문자열과 자료구조 실습

이 노트북은 03-1의 자료형과 형변환을 짧게 적용한 뒤 문자열 처리와 `list`, `tuple`, `set`, `dict`에 집중합니다. 조건문과 반복문 없이 모든 필수 셀을 실행할 수 있습니다.

## 1. 문자열 인덱싱과 슬라이싱

실행 전에 각 출력 결과를 예상하세요.

In [ ]:
action = "DENY"
timestamp = "2026-08-26T13:45:10Z"

print(action[0])
print(action[-1])
print(timestamp[:10])
print(timestamp[11:19])
print(action[0:100])

# IndexError를 관찰하려면 주석을 제거하세요.
# print(action[4])

## 2. 원본 보존과 문자열 정규화

문자열 메서드는 원본을 바꾸지 않고 새 문자열을 반환합니다.

In [ ]:
raw_line = "  Deny 198.51.100.9 /Admin\n"
clean_line = raw_line.strip()
normalized_line = clean_line.lower()

print(repr(raw_line))
print(repr(clean_line))
print(normalized_line)
print(normalized_line.startswith("deny"))
print("/admin" in normalized_line)

## 3. split, partition, join, f-string

문자열을 필드로 분리하고 다시 결합합니다.

In [ ]:
line = "DENY 198.51.100.9 443 /admin"
parts = line.split()
action, ip, port_text, path = parts
port = int(port_text)

print(parts)
print(" | ".join(parts))
print(f"action={action} ip={ip} port={port} path={path}")

header = "Content-Type: application/json"
name, separator, value = header.partition(":")
print(name, separator, value.strip())

## 4. list와 tuple

리스트는 변경할 수 있고, 튜플의 각 위치는 다른 값으로 재대입할 수 없습니다.

In [ ]:
ports = [22, 80]
ports.append(443)
ports.extend([8080, 8443])
print(ports)
print(sorted(ports))

endpoint = ("tcp", "198.51.100.9", 443)
protocol, endpoint_ip, endpoint_port = endpoint
print(protocol, endpoint_ip, endpoint_port)
print(type((443)))
print(type((443,)))

## 5. set과 dict

세트는 고유값과 집합 비교에, 딕셔너리는 이름이 붙은 속성에 사용합니다.

In [ ]:
observed_ips = {"10.0.0.5", "198.51.100.9", "10.0.0.5"}
blocked_ips = {"198.51.100.9", "203.0.113.7"}

print(observed_ips)
print(observed_ips & blocked_ips)
print(type(set()))
print(type({}))

event = {
    "action": "DENY",
    "ip": "198.51.100.9",
    "port": 443,
    "path": "/admin",
}
print(event["ip"])
print(event.get("country"))
print(event.get("severity", "UNKNOWN"))

## 6. list[dict]와 중첩 구조

왼쪽부터 한 단계씩 자료구조를 따라갑니다.

In [ ]:
events = [
    {"action": "ALLOW", "ip": "10.0.0.5", "port": 80},
    {"action": "DENY", "ip": "198.51.100.9", "port": 443},
]

report = {
    "summary": {"total": 2, "deny": 1},
    "events": events,
}

print(events[1]["ip"])
print(report["summary"]["deny"])
print(report["events"][0]["port"])

## 7. 별칭, 얕은 복사, 깊은 복사

각 결과에서 바깥 딕셔너리와 내부 리스트 중 무엇이 공유되는지 설명하세요.

In [ ]:
from copy import deepcopy

alias_original = {"action": "DENY", "tags": ["auth"]}
alias = alias_original
alias["action"] = "REVIEW"
print("alias original:", alias_original)

shallow_original = {"action": "DENY", "tags": ["auth"]}
shallow = shallow_original.copy()
shallow["action"] = "REVIEW"
shallow["tags"].append("critical")
print("shallow original:", shallow_original)
print("shallow copy:", shallow)

deep_original = {"action": "DENY", "tags": ["auth"]}
deep = deepcopy(deep_original)
deep["action"] = "REVIEW"
deep["tags"].append("critical")
print("deep original:", deep_original)
print("deep copy:", deep)

## 8. 미니 실습 — 세 로그 구조화

조건문과 반복문을 사용하지 않고 세 로그를 `list[dict]`로 변환합니다. 반복되는 코드는 03-4 학습 후 개선합니다.

In [ ]:
lines = [
    "ALLOW 10.0.0.5 80 /index",
    "DENY 198.51.100.9 443 /admin",
    "DENY 198.51.100.9 443 /login",
]

parts_1 = lines[0].split()
parts_2 = lines[1].split()
parts_3 = lines[2].split()

event_1 = {"action": parts_1[0], "ip": parts_1[1], "port": int(parts_1[2]), "path": parts_1[3]}
event_2 = {"action": parts_2[0], "ip": parts_2[1], "port": int(parts_2[2]), "path": parts_2[3]}
event_3 = {"action": parts_3[0], "ip": parts_3[1], "port": int(parts_3[2]), "path": parts_3[3]}

events = [event_1, event_2, event_3]
unique_ips = {event_1["ip"], event_2["ip"], event_3["ip"]}
actions = [event_1["action"], event_2["action"], event_3["action"]]
deny_count = actions.count("DENY")

report = {
    "summary": {"total": len(events), "deny": deny_count},
    "unique_ips": unique_ips,
    "events": events,
}

print(report)

### 자기점검

아래 셀이 오류 없이 끝나면 필수 실습을 완료한 것입니다.

In [ ]:
assert len(events) == 3
assert events[0]["port"] == 80
assert type(events[1]["port"]) is int
assert events[2]["path"] == "/login"
assert unique_ips == {"10.0.0.5", "198.51.100.9"}
assert deny_count == 2
assert report["summary"]["total"] == 3
assert report["summary"]["deny"] == 2

print("모든 자기점검을 통과했습니다.")

## 9. 마무리 확인

다음을 코드와 말로 설명할 수 있는지 확인하세요.

- 문자열 인덱싱과 슬라이싱의 경계 차이
- 원본과 정규화 문자열을 분리하는 이유
- `append()`와 `extend()`, `sort()`와 `sorted()`의 차이
- 빈 세트가 `{}`가 아닌 `set()`인 이유
- `dict[key]`와 `dict.get(key)`의 차이
- 별칭, 얕은 복사, 깊은 복사의 공유 범위

다음 절에서는 조건문을 사용해 이벤트를 허용·차단·오류로 분류합니다.